In [2]:
import os

In [3]:
%pwd

'c:\\Users\\HP\\Desktop\\Python\\Vs_Python\\MLoPs\\Movie_recommendation_system\\Notebook'

In [4]:
os.chdir('../')

In [5]:
%pwd

'c:\\Users\\HP\\Desktop\\Python\\Vs_Python\\MLoPs\\Movie_recommendation_system'

In [6]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class ModelTrainerConfig:
    root_dir: Path
    transformed_data_path: Path
    vectorizer_path: Path
    similarity_path: Path

In [7]:
from Movie_Recommendation_system.constants import *
from Movie_Recommendation_system.utils.common import read_yaml, create_directories

In [8]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH,
        schema_filepath = SCHEMA_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])

    def get_model_trainer_config(self) -> ModelTrainerConfig:
        config = self.config.model_trainer

        create_directories([Path(config.root_dir)])

        model_trainer_config = ModelTrainerConfig(
            root_dir=config.root_dir,
            transformed_data_path=config.transformed_data_path,
            vectorizer_path=config.vectorizer_path,
            similarity_path=config.similarity_path,
        )

        return model_trainer_config

In [9]:
import pandas as pd
import pickle
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from Movie_Recommendation_system import logger

class ModelTrainer:
    def __init__(self, config: ModelTrainerConfig):
        self.config = config

    def build_model(self):
        # 1️⃣ Load transformed data
        movies = pd.read_csv(self.config.transformed_data_path)

        logger.info('Load Transformed data')

        # 2️⃣ Vectorize tags
        cv = CountVectorizer(max_features=5000, stop_words="english")
        vectors = cv.fit_transform(movies["tags"]).toarray()

        # 3️⃣ Compute cosine similarity
        similarity = cosine_similarity(vectors)

        logger.info('Compute cosine similarity')

        # 4️⃣ Save vectorizer
        with open(self.config.vectorizer_path, "wb") as f:
            pickle.dump(cv, f)

        logger.info('Save Vectorizer')

        # 5️⃣ Save similarity matrix
        with open(self.config.similarity_path, "wb") as f:
            pickle.dump(similarity, f)

        logger.info('Save similarity matrix')

        print("✅ Model build completed successfully")


In [10]:
try:
    config = ConfigurationManager()
    model_trainer_config = config.get_model_trainer_config()

    model_trainer_config = ModelTrainer(config = model_trainer_config)
    model_trainer_config.build_model()

except Exception as e:
    raise e


[2026-02-11 15:36:45,285: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-02-11 15:36:45,296: INFO: common: yaml file: params.yaml loaded successfully]
[2026-02-11 15:36:45,311: INFO: common: yaml file: schema.yaml loaded successfully]
[2026-02-11 15:36:45,316: INFO: common: created directory at: artifacts]
[2026-02-11 15:36:45,320: INFO: common: created directory at: artifacts\model_trainer]


[2026-02-11 15:36:45,447: INFO: 2919082751: Load Transformed data]
[2026-02-11 15:36:49,701: INFO: 2919082751: Compute cosine similarity]
[2026-02-11 15:36:49,753: INFO: 2919082751: Save Vectorizer]
[2026-02-11 15:36:50,211: INFO: 2919082751: Save similarity matrix]
✅ Model build completed successfully
